In [2]:
!pip install pandas numpy scikit-learn tensorflow joblib


In [5]:
# ==========================================================
# OS LOGS MLP TRAINING PIPELINE
# ==========================================================

import os
import random
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow import keras
from tensorflow.keras import layers


# ==========================================================
# Reproducibility
# ==========================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


# ==========================================================
# Load Dataset
# ==========================================================

csv_path = "/content/drive/MyDrive/Colab Notebooks/OS_Training_Ready_Blind.csv.gz"

df = pd.read_csv(csv_path, compression="gzip", low_memory=False)
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)


# ==========================================================
# Target Label
# ==========================================================

df["Label"] = df["Malicious_Label"]


# ==========================================================
# Feature Selection (OS Logs)
# ==========================================================

selected_features = [
    "_source.agent.ip",
    "_source.data.audit.exe",
    "_source.data.audit.file.name",
    "_source.data.audit.type",
    "executed_command",
    "effective_uid",
    "hour_sin",
    "hour_cos"
]

joblib.dump(
    selected_features,
    "/content/drive/MyDrive/Colab Notebooks/mlp_os_feature_columns.pkl"
)

df = df[selected_features + ["Label"]]


# ==========================================================
# Handle Missing Values
# ==========================================================

df = df.fillna("Unknown")


# ==========================================================
# Encode Categorical Columns
# ==========================================================

categorical_cols = [
    "_source.agent.ip",
    "_source.data.audit.exe",
    "_source.data.audit.file.name",
    "_source.data.audit.type",
    "executed_command"
]

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

joblib.dump(
    label_encoders,
    "/content/drive/MyDrive/Colab Notebooks/mlp_os_label_encoders.pkl"
)


# ==========================================================
# Convert Numeric Columns
# ==========================================================

numeric_cols = ["effective_uid", "hour_sin", "hour_cos"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols] = df[numeric_cols].fillna(0)


# ==========================================================
# Train Test Split
# ==========================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["Label"]
)

X_train = train_df[selected_features]
X_test = test_df[selected_features]

y_train = train_df["Label"]
y_test = test_df["Label"]


# ==========================================================
# Feature Scaling
# ==========================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(
    scaler,
    "/content/drive/MyDrive/Colab Notebooks/mlp_os_scaler.pkl"
)


# ==========================================================
# Handle Class Imbalance
# ==========================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))

print("Class Weights:", class_weight_dict)


# ==========================================================
# Build MLP Model
# ==========================================================

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


# ==========================================================
# Train Model
# ==========================================================

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=4096,
    validation_split=0.2,
    class_weight=class_weight_dict
)


# ==========================================================
# Save Model
# ==========================================================

model.save(
    "/content/drive/MyDrive/Colab Notebooks/mlp_os_model.h5"
)

print("MLP OS model saved successfully")


# ==========================================================
# Evaluation
# ==========================================================

y_prob = model.predict(X_test).ravel()

y_pred = (y_prob > 0.3).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# ==========================================================
# END OF PIPELINE
# ==========================================================

Dataset shape: (122563, 160)
Class Weights: {0: np.float64(0.6267017781584363), 1: np.float64(2.4731372647934218)}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,289 (48.00 KB)

 Trainable params: 11,905 (46.50 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 74ms/step - accuracy: 0.5442 - loss: 0.6604 - val_accuracy: 0.8733 - val_loss: 0.5000
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.8638 - loss: 0.2794 - val_accuracy: 0.9086 - val_loss: 0.3473
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.8790 - loss: 0.2357 - val_accuracy: 0.9083 - val_loss: 0.2739
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.8799 - loss: 0.2197 - val_accuracy: 0.9083 - val_loss: 0.2362
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.8798 - loss: 0.2152 - val_accuracy: 0.8778 - val_loss: 0.2149
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.8795 - loss: 0.2094 - val_accuracy: 0.8773 - val_loss: 0.2022
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.8796 - loss: 0.2037 - val_accuracy: 0.8773 - val_loss: 0.1924
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.8802 - loss: 0.1981 - val_accuracy: 0.8775 - v

MLP OS model saved successfully
767/767 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Confusion Matrix:
[[16751  2806]
 [   37  4919]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.86      0.92     19557
           1       0.64      0.99      0.78      4956

    accuracy                           0.88     24513
   macro avg       0.82      0.92      0.85     24513
weighted avg       0.92      0.88      0.89     24513

